<div style="background-color:rgb(0, 55, 207); padding: 30px; border-radius: 20px; box-shadow: 0 4px 15px rgba(105, 195, 255, 0.3); color:rgb(187, 201, 248); font-family: 'Times New Roman', serif;">

<h1 style="text-align: center; font-size: 38px; color: white; font-weight: bold;">Training CNN-LSTM</h1>

<h3 style="font-size: 22px; color: white; font-weight: bold;">Libraries</h3>

In [ ]:

import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.model_selection import StratifiedKFold
from torch.nn.utils.rnn import pad_sequence

<h3 style="font-size: 22px; color: white; font-weight: bold;">Dataset</h3>

In [3]:

class SignLanguageDataset(Dataset):
    def __init__(self, video_folder, excel_path, indices):
        self.data = pd.read_excel(excel_path)
        self.data['video_id'] = self.data['video_id'].apply(lambda x: str(x).zfill(5))
        self.data = self.data.iloc[indices].reset_index(drop=True)

        self.video_folder = video_folder
        self.label2idx = {label: idx for idx, label in enumerate(sorted(self.data['gloss'].unique()))}
        self.idx2label = {v: k for k, v in self.label2idx.items()}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        video_id = row['video_id']
        gloss = row['gloss']

        path = os.path.join(self.video_folder, f"{video_id}.npy")
        keypoints = np.load(path)

        keypoints = torch.tensor(keypoints, dtype=torch.float32)
        label = torch.tensor(self.label2idx[gloss], dtype=torch.long)

        return keypoints, label


<h3 style="font-size: 22px; color: white; font-weight: bold;">Model Architecture</h3>

In [ ]:
class SignLanguageLSTM(nn.Module):
    def __init__(self, num_classes, input_size=126, hidden_size=128, dropout=0.5):
        super(SignLanguageLSTM, self).__init__()

        # CNN
        self.conv1 = nn.Conv1d(in_channels=input_size, out_channels=128, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(128)
        self.conv2 = nn.Conv1d(in_channels=128, out_channels=128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(128)
        self.relu = nn.ReLU()
        self.dropout_conv = nn.Dropout(p=dropout)

        # LSTM
        self.lstm1 = nn.LSTM(input_size=128, hidden_size=hidden_size, batch_first=True, bidirectional=True)
        self.norm1 = nn.LayerNorm(hidden_size * 2)
        self.dropout_lstm = nn.Dropout(p=dropout)

        self.lstm2 = nn.LSTM(input_size=hidden_size * 2, hidden_size=hidden_size, batch_first=True, bidirectional=True)
        self.norm2 = nn.LayerNorm(hidden_size * 2)

        self.dropout_fc = nn.Dropout(p=dropout)
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = x.permute(0, 2, 1) 
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.dropout_conv(x)
        x = x.permute(0, 2, 1)
        x, _ = self.lstm1(x)
        x = self.norm1(x)
        x = self.dropout_lstm(x)

        x, _ = self.lstm2(x)
        x = self.norm2(x)

        x = torch.max(x, dim=1)[0] 
        x = self.dropout_fc(x)
        x = self.fc2(x)
        return x

<h3 style="font-size: 22px; color: white; font-weight: bold;">Data Padding</h3>

In [ ]:
def pad_collate(batch):
    sequences, labels = zip(*batch)
    padded_sequences = pad_sequence(sequences, batch_first=True)  # auto-pads to max length
    labels = torch.tensor(labels)
    return padded_sequences, labels


<h3 style="font-size: 22px; color: white; font-weight: bold;">Configuration</h3>

In [ ]:
# === Configuration ===
video_dir = r"PATH_TO_YOUR_NUMPY_FILES"
excel_path = r"PATH_TO_YOUR_LABELS_EXCEL"
batch_size = 32
epochs = 2000
early_stopping_acc = 90.0

<h3 style="font-size: 22px; color: white; font-weight: bold;">Load Data</h3>

In [ ]:
all_data = pd.read_excel(excel_path)
all_data['video_id'] = all_data['video_id'].apply(lambda x: str(x).zfill(5))
label_map = {label: idx for idx, label in enumerate(sorted(all_data['gloss'].unique()))}
y = all_data['gloss'].map(label_map).values

<h3 style="font-size: 22px; color: white; font-weight: bold;">Cross-validation</h3>

In [ ]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
fold = 1

for train_idx, val_idx in skf.split(np.zeros(len(y)), y):
    print(f"\n🔁 Fold {fold}")
    
    train_set = SignLanguageDataset(video_dir, excel_path, train_idx)
    val_set = SignLanguageDataset(video_dir, excel_path, val_idx)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, collate_fn=pad_collate)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, collate_fn=pad_collate)

    model = SignLanguageLSTM(num_classes=len(label_map)).to("cuda" if torch.cuda.is_available() else "cpu")
    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        train_loss, correct, total = 0, 0, 0

        for x, y_true in train_loader:
            x, y_true = x.cuda(), y_true.cuda()
            optimizer.zero_grad()
            output = model(x)
            loss = criterion(output, y_true)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            correct += (output.argmax(1) == y_true).sum().item()
            total += y_true.size(0)

        train_acc = 100 * correct / total
        print(f"📌 Fold {fold} | Epoch {epoch+1}: Loss = {train_loss:.4f}, Accuracy = {train_acc:.2f}%")

        # Early stopping condition
        if train_acc >= early_stopping_acc:
            print(f"✅ Early stopping triggered at epoch {epoch+1} with accuracy {train_acc:.2f}%\n")
            break

    fold += 1

<h3 style="font-size: 22px; color: white; font-weight: bold;">Evaluation</h3>

In [ ]:
# Evaluate on validation set
model.eval()
correct = 0
total = 0


with torch.no_grad():
    for x, y_true in val_loader:
        x, y_true = x.cuda(), y_true.cuda()
        output = model(x)
        preds = torch.argmax(output, dim=1)
        correct += (preds == y_true).sum().item()
        total += y_true.size(0)

val_acc = 100 * correct / total
print(f"✅ Fold {fold} Validation Accuracy: {val_acc:.2f}%")

# Save model
torch.save(model.state_dict(), f"best_model_fold_{fold}.pth")
print(f"💾 Model saved as best_model_fold_{fold}.pth")